# Lab — Basic RAG

**Objective:** Wire retrieve → context → answer stages without an external LLM API.

**Book track:** `06-knowledge-and-retrieval-systems` · **Time:** 30–45 minutes · **Python:** 3.10+

Work through the cells in order. Each code cell should run top-to-bottom. Keep `main.py` in this folder aligned with your final answers—`pytest` validates that file.


## How to use this notebook

1. Open from the lab directory (`labs/03-basic-rag/`) in Jupyter, VS Code, or Codespaces.
2. Run cells sequentially; restart the kernel if you change earlier definitions.
3. Complete **Your turn** sections, then sync working code into `main.py`.
4. Run the verification cell (`pytest`) before you finish.


In [ ]:
from pathlib import Path

LAB_DIR = Path('.').resolve()
assert (LAB_DIR / 'main.py').exists(), (
    'Start Jupyter from the lab directory, e.g. labs/03-basic-rag/'
)
print('Lab directory:', LAB_DIR)


## Tasks

1. Trace retrieval scores for a query with no lexical overlap.
2. Add an abstention path when no evidence passes threshold.
3. Verify citations appear only when evidence is used.
4. Compare answer quality with k=1 vs k=2 retrieval.


## Spec-first checkpoint

Before the coding cells below, list acceptance cases for **Basic RAG** (`labs/03-basic-rag/`):

| Case | Input / setup | Expected outcome |
|---|---|---|
| Normal | … | … |
| Boundary | … | … |
| Adversarial | … | … |

**Cursor:** `@specs/03-basic-rag.yaml` or paste the table into Plan mode.  
**OpenSpec:** `/opsx:propose` → review `proposal.md` + delta specs → `/opsx:apply` after approval.

Sync passing code into `main.py` before running pytest.


## Step 1 — Evidence store

RAG separates **retrieval** (find evidence) from **generation** (compose an answer). Here generation is template-based so you can inspect each stage.


In [ ]:
import re

PASSAGES = [
    ('leave', 'Employees receive 20 days of annual leave per calendar year.'),
    ('expenses', 'Expense claims must be submitted within 30 days of purchase.'),
    ('security', 'Suspected credential exposure must be reported immediately.'),
]


def words(text: str) -> set[str]:
    return set(re.findall(r'[a-z0-9]+', text.lower()))


for source, text in PASSAGES:
    print(f'[{source}]', text)


## Step 2 — Retrieve top-k passages by lexical overlap


In [ ]:
def retrieve(question: str, k: int = 2) -> list[tuple[int, str, str]]:
    q = words(question)
    scored = [(len(q & words(text)), source, text) for source, text in PASSAGES]
    return sorted(scored, reverse=True)[:k]


question = 'How soon must I submit an expense claim?'
print('Question:', question)
print('Retrieved:', retrieve(question))


## Step 3 — Ground answers in evidence with citations


In [ ]:
def answer(question: str) -> str:
    evidence = [item for item in retrieve(question) if item[0] > 0]
    if not evidence:
        return 'I do not have relevant evidence to answer that question.'
    citations = ' '.join(f'[{source}]' for _, source, _ in evidence)
    context = ' '.join(text for _, _, text in evidence)
    return f'Evidence: {context} {citations}'


print('Answer:', answer(question))


## Your turn

1. Trace retrieval for a query with **no lexical overlap**.
2. Confirm the **abstention** message when nothing matches.
3. Verify citations appear only when evidence is used.
4. Compare `k=1` vs `k=2` retrieval quality.

Update `main.py` with your final functions.


In [ ]:
off_topic = 'What is the stock price of the company?'
print('Retrieve:', retrieve(off_topic))
print('Answer:', answer(off_topic))

print('k=1:', answer(question))  # temporarily set k=1 in retrieve to compare


## Verify

Run the test suite against `main.py` and `test_lab.py`.


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'test_lab.py', '-q'],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
assert result.returncode == 0, 'Tests failed—see output above'


## Reflection

- What broke first when you changed inputs?
- Which simpler baseline would you compare against in a design review?

## Extensions

- Add another case to `test_lab.py`.
- Link observations to a concept card on the AIEBOK site.
